In [21]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import os

# Create Spark session
spark = SparkSession.builder \
    .appName("Bluebikes Data Exploration") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark session created!")

Spark session created!


In [22]:
spark.sparkContext.setLogLevel("WARN")

In [23]:
#Paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
PATH_2023 = os.path.join(BASE_DIR, "data/raw/2023/*.csv")
PATH_2024 = os.path.join(BASE_DIR, "data/raw/2024/*.csv")

In [24]:
print("📥 Loading 2023 data...")
df_2023 = spark.read.csv(PATH_2023, header=True, inferSchema=True)
count_2023 = df_2023.count()
print(f"✅ 2023: {count_2023:,} records")

📥 Loading 2023 data...


26/01/28 10:58:55 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/raw/2023/*.csv.
java.io.FileNotFoundException: File /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/raw/2023/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst

✅ 2023: 3,693,193 records


In [25]:
print("\n📥 Loading 2024 data...")
df_2024 = spark.read.csv(PATH_2024, header=True, inferSchema=True)
count_2024 = df_2024.count()
print(f"✅ 2024: {count_2024:,} records")


📥 Loading 2024 data...


26/01/28 10:59:03 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/raw/2024/*.csv.
java.io.FileNotFoundException: File /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/raw/2024/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst

✅ 2024: 4,751,790 records


In [26]:
print(f"\n📊 Total: {count_2023 + count_2024:,} records")


📊 Total: 8,444,983 records


In [ ]:
# Compare schemas between 2023 and 2024
print("="*80)
print("🔍 SCHEMA COMPARISON: 2023 vs 2024")
print("="*80)

# Get column names
cols_2023 = set(df_2023.columns)
cols_2024 = set(df_2024.columns)

print(f"\n 2023 has {len(cols_2023)} columns")
print(f" 2024 has {len(cols_2024)} columns")

# Check if identical
if cols_2023 == cols_2024:
    print("Column names are IDENTICAL!")
else:
    print(" Column names are DIFFERENT!")

    # Show differences
    only_2023 = cols_2023 - cols_2024
    only_2024 = cols_2024 - cols_2023

    if only_2023:
        print(f"\n Only in 2023: {sorted(only_2023)}")

    if only_2024:
        print(f"\n Only in 2024: {sorted(only_2024)}")

# Show common columns
common_cols = sorted(cols_2023 & cols_2024)
print(f"\n Common columns ({len(common_cols)}):")
for i, col_name in enumerate(common_cols, 1):
    print(f"  {i}. {col_name}")

🔍 SCHEMA COMPARISON: 2023 vs 2024

 2023 has 13 columns
 2024 has 13 columns
Column names are IDENTICAL!

 Common columns (13):
  1. end_lat
  2. end_lng
  3. end_station_id
  4. end_station_name
  5. ended_at
  6. member_casual
  7. ride_id
  8. rideable_type
  9. start_lat
  10. start_lng
  11. start_station_id
  12. start_station_name
  13. started_at


In [30]:
# Compare data types for each column
print("\n" + "="*80)
print("🔍 DATA TYPE COMPARISON")
print("="*80)

# Get schemas
schema_2023 = {field.name: str(field.dataType) for field in df_2023.schema.fields}
schema_2024 = {field.name: str(field.dataType) for field in df_2024.schema.fields}

# Compare data types for common columns
print(f"\n{'Column':<25} {'2023 Type':<20} {'2024 Type':<20} {'Match'}")
print("-"*80)

for col_name in sorted(common_cols):
    type_2023 = schema_2023[col_name]
    type_2024 = schema_2024[col_name]
    match = "✅" if type_2023 == type_2024 else "❌"
    print(f"{col_name:<25} {type_2023:<20} {type_2024:<20} {match}")


🔍 DATA TYPE COMPARISON

Column                    2023 Type            2024 Type            Match
--------------------------------------------------------------------------------
end_lat                   StringType()         DoubleType()         ❌
end_lng                   StringType()         DoubleType()         ❌
end_station_id            StringType()         StringType()         ✅
end_station_name          StringType()         StringType()         ✅
ended_at                  StringType()         TimestampType()      ❌
member_casual             StringType()         StringType()         ✅
ride_id                   StringType()         StringType()         ✅
rideable_type             StringType()         StringType()         ✅
start_lat                 StringType()         DoubleType()         ❌
start_lng                 StringType()         DoubleType()         ❌
start_station_id          StringType()         StringType()         ✅
start_station_name        StringType()         Str

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType

# Define the schema we WANT (proper types)
schema = StructType([
    StructField("ride_id", StringType(), True),
    StructField("rideable_type", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("start_station_name", StringType(), True),
    StructField("start_station_id", StringType(), True),
    StructField("end_station_name", StringType(), True),
    StructField("end_station_id", StringType(), True),
    StructField("start_lat", DoubleType(), True),
    StructField("start_lng", DoubleType(), True),
    StructField("end_lat", DoubleType(), True),
    StructField("end_lng", DoubleType(), True),
    StructField("member_casual", StringType(), True)
])


In [ ]:
# Reload both years with explicit schema
print("📥 Reloading data with explicit schema...")

df_2023 = spark.read.csv(
    PATH_2023,
    header=True,
    schema=schema,
    timestampFormat="yyyy-MM-dd HH:mm:ss"  # Handle timestamp format
)

df_2024 = spark.read.csv(
    PATH_2024,
    header=True,
    schema=schema,
    timestampFormat="yyyy-MM-dd HH:mm:ss"
)

print(f"✅ 2023: {df_2023.count():,} records")
print(f"✅ 2024: {df_2024.count():,} records")

# Verify types are now consistent
print("\n🔍 Checking data types...")
print(f"\n2023 Schema:")
df_2023.printSchema()

print(f"\n2024 Schema:")
df_2024.printSchema()

📥 Reloading data with explicit schema...


26/01/28 11:03:20 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/raw/2023/*.csv.
java.io.FileNotFoundException: File /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/raw/2023/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst

✅ 2023: 3,693,193 records
✅ 2024: 4,751,790 records

🔍 Checking data types...

2023 Schema:
root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)


2024 Schema:
root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = tr

In [ ]:
# Check for NULL/missing values in both years
print("="*80)
print("🔍 MISSING VALUES ANALYSIS")
print("="*80)

from pyspark.sql.functions import col, count, when, isnan, isnull

def check_nulls(df, year_name):
    print(f"\n📅 {year_name}")
    print("-"*80)

    total = df.count()

    # Count nulls in each column
    null_counts = {}
    for col_name in df.columns:
        null_count = df.filter(col(col_name).isNull()).count()
        if null_count > 0:
            null_pct = (null_count / total) * 100
            null_counts[col_name] = (null_count, null_pct)

    if null_counts:
        for col_name, (count, pct) in sorted(null_counts.items(), key=lambda x: x[1][0], reverse=True):
            print(f"  {col_name:<25} {count:>10,} ({pct:>6.2f}%)")
    else:
        print("  ✅ No NULL values found!")

    return null_counts

nulls_2023 = check_nulls(df_2023, "2023")
nulls_2024 = check_nulls(df_2024, "2024")

🔍 MISSING VALUES ANALYSIS

📅 2023
--------------------------------------------------------------------------------


26/01/28 11:06:52 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: tripduration
 Schema: ride_id
Expected: ride_id but found: tripduration
CSV file: file:///Users/anshumansingh/Machine%20Learning%20/bluebikes-demand-predictor/data/raw/2023/202303-bluebikes-tripdata.csv
26/01/28 11:06:52 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: tripduration
 Schema: ride_id
Expected: ride_id but found: tripduration
CSV file: file:///Users/anshumansingh/Machine%20Learning%20/bluebikes-demand-predictor/data/raw/2023/202302-bluebikes-tripdata.csv
26/01/28 11:06:52 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: tripduration
 Schema: ride_id
Expected: ride_id but found: tripduration
CSV file: file:///Users/anshumansingh/Machine%20Learning%20/bluebikes-demand-predictor/data/raw/2023/202301-bluebikes-tripdata.csv
26/01/28 11:06:52 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: starttime
 Schema: 

  started_at                   492,318 ( 13.33%)
  ended_at                     492,318 ( 13.33%)
  start_lat                    492,318 ( 13.33%)
  end_station_name              17,245 (  0.47%)
  end_station_id                17,245 (  0.47%)
  end_lat                       17,166 (  0.46%)
  end_lng                       17,166 (  0.46%)
  start_station_name                 8 (  0.00%)
  start_station_id                   8 (  0.00%)

📅 2024
--------------------------------------------------------------------------------


  started_at                 3,294,596 ( 69.33%)
  ended_at                   3,294,596 ( 69.33%)
  end_station_id                 8,628 (  0.18%)
  end_station_name               8,225 (  0.17%)
  end_lat                        3,507 (  0.07%)
  end_lng                        3,507 (  0.07%)
  start_station_name             1,435 (  0.03%)
  start_station_id               1,435 (  0.03%)


In [ ]:
# Check schema for each month in 2023
print("="*80)
print("🔍 CHECKING EACH MONTH IN 2023")
print("="*80)

import glob

months_2023 = sorted(glob.glob(os.path.join(BASE_DIR, "data/raw/2023/*.csv")))

for month_file in months_2023:
    month_name = os.path.basename(month_file)

    # Read just this file
    df_month = spark.read.csv(month_file, header=True, inferSchema=True)

    # Check first column name
    first_col = df_month.columns[0]
    schema_type = "OLD" if first_col == "tripduration" else "NEW"

    print(f"{month_name[:6]}: {schema_type} format - Columns: {df_month.columns[:3]}")

🔍 CHECKING EACH MONTH IN 2023
202301: OLD format - Columns: ['tripduration', 'starttime', 'stoptime']
202302: OLD format - Columns: ['tripduration', 'starttime', 'stoptime']
202303: OLD format - Columns: ['tripduration', 'starttime', 'stoptime']
202304: NEW format - Columns: ['ride_id', 'rideable_type', 'started_at']
202305: NEW format - Columns: ['ride_id', 'rideable_type', 'started_at']
202306: NEW format - Columns: ['ride_id', 'rideable_type', 'started_at']
202307: NEW format - Columns: ['ride_id', 'rideable_type', 'started_at']
202308: NEW format - Columns: ['ride_id', 'rideable_type', 'started_at']
202309: NEW format - Columns: ['ride_id', 'rideable_type', 'started_at']
202310: NEW format - Columns: ['ride_id', 'rideable_type', 'started_at']
202311: NEW format - Columns: ['ride_id', 'rideable_type', 'started_at']
202312: NEW format - Columns: ['ride_id', 'rideable_type', 'started_at']


In [ ]:
# Load only NEW format months
print("="*80)
print("📥 Loading NEW FORMAT Data Only")
print("="*80)

# 2023: April onwards (months 04-12)
months_new_2023 = [f"202304", "202305", "202306", "202307", "202308",
                   "202309", "202310", "202311", "202312"]

files_2023_new = [os.path.join(BASE_DIR, f"data/raw/2023/{month}-bluebikes-tripdata.csv")
                  for month in months_new_2023]

# Load 2023 new format
df_2023_new = spark.read.csv(files_2023_new, header=True, inferSchema=True)

# Load all 2024
df_2024_all = spark.read.csv(
    os.path.join(BASE_DIR, "data/raw/2024/*.csv"),
    header=True,
    inferSchema=True
)

print(f"✅ 2023 (Apr-Dec): {df_2023_new.count():,} records")
print(f"✅ 2024 (All):     {df_2024_all.count():,} records")
print(f"📊 Total:          {df_2023_new.count() + df_2024_all.count():,} records")

# Verify schemas match
if set(df_2023_new.columns) == set(df_2024_all.columns):
    print("\n✅ Schemas are IDENTICAL!")
else:
    print("\n❌ Schemas still differ!")

📥 Loading NEW FORMAT Data Only


26/01/28 11:09:20 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/raw/2024/*.csv.
java.io.FileNotFoundException: File /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/raw/2024/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst

✅ 2023 (Apr-Dec): 3,200,875 records
✅ 2024 (All):     4,751,790 records
📊 Total:          7,952,665 records

✅ Schemas are IDENTICAL!


In [ ]:
# Check for NULL values
print("="*80)
print("🔍 MISSING VALUES ANALYSIS")
print("="*80)

def check_missing(df, name):
    print(f"\n📅 {name}")
    print("-"*80)

    total = df.count()
    has_nulls = False

    for col_name in df.columns:
        null_count = df.filter(col(col_name).isNull()).count()
        if null_count > 0:
            has_nulls = True
            pct = (null_count / total) * 100
            print(f"  {col_name:<25} {null_count:>10,} ({pct:>6.2f}%)")

    if not has_nulls:
        print("  ✅ No NULL values!")

check_missing(df_2023_new, "2023 (Apr-Dec)")
check_missing(df_2024_all, "2024")

🔍 MISSING VALUES ANALYSIS

📅 2023 (Apr-Dec)
--------------------------------------------------------------------------------
  start_station_name                 8 (  0.00%)
  start_station_id                   8 (  0.00%)
  end_station_name              17,245 (  0.54%)
  end_station_id                17,245 (  0.54%)
  end_lat                       17,166 (  0.54%)
  end_lng                       17,166 (  0.54%)

📅 2024
--------------------------------------------------------------------------------


  start_station_name             1,435 (  0.03%)


  start_station_id               1,435 (  0.03%)


  end_station_name               8,225 (  0.17%)


  end_station_id                 8,628 (  0.18%)


  end_lat                        3,507 (  0.07%)


  end_lng                        3,507 (  0.07%)


In [ ]:
# Check duplicates
print("\n" + "="*80)
print("🔍 DUPLICATE ANALYSIS")
print("="*80)

def check_dupes(df, name):
    print(f"\n📅 {name}")
    print("-"*80)

    total = df.count()
    unique = df.select('ride_id').distinct().count()
    dupes = total - unique

    print(f"  Total records:   {total:,}")
    print(f"  Unique ride_ids: {unique:,}")
    print(f"  Duplicates:      {dupes:,} ({dupes/total*100:.2f}%)")

    if dupes > 0:
        print("\n  Sample duplicate ride_ids:")
        df.groupBy('ride_id').count() \
          .filter(col('count') > 1) \
          .orderBy(col('count').desc()) \
          .show(5)

check_dupes(df_2023_new, "2023 (Apr-Dec)")
check_dupes(df_2024_all, "2024")


🔍 DUPLICATE ANALYSIS

📅 2023 (Apr-Dec)
--------------------------------------------------------------------------------


  Total records:   3,200,875
  Unique ride_ids: 3,200,875
  Duplicates:      0 (0.00%)

📅 2024
--------------------------------------------------------------------------------


  Total records:   4,751,790
  Unique ride_ids: 4,751,649
  Duplicates:      141 (0.00%)

  Sample duplicate ride_ids:


+----------------+-----+
|         ride_id|count|
+----------------+-----+
|521E96AE8D764105|    2|
|F5F8A07930818F7E|    2|
|64EDB331204AAD2D|    2|
|13FCDE02B167ADC2|    2|
|322A2ED0411EFCF2|    2|
+----------------+-----+
only showing top 5 rows


In [44]:
# Analyze trip durations
print("\n" + "="*80)
print("🔍 TRIP DURATION ANALYSIS")
print("="*80)

def analyze_duration(df, name):
    print(f"\n📅 {name}")
    print("-"*80)

    # Calculate duration in seconds
    df_temp = df.withColumn(
        'duration_sec',
        unix_timestamp('ended_at') - unix_timestamp('started_at')
    )

    # Statistics
    print("\n  Duration Statistics:")
    duration_stats = df_temp.select('duration_sec').describe().collect()

    for row in duration_stats:
        stat = row[0]
        value_sec = float(row[1]) if row[1] else 0
        value_min = value_sec / 60
        print(f"    {stat:<10} {value_sec:>12,.0f} sec ({value_min:>8,.1f} min)")

    # Categorize
    total = df_temp.count()

    negative = df_temp.filter(col('duration_sec') < 0).count()
    under_1min = df_temp.filter((col('duration_sec') >= 0) & (col('duration_sec') < 60)).count()
    normal = df_temp.filter((col('duration_sec') >= 60) & (col('duration_sec') <= 86400)).count()
    over_24h = df_temp.filter(col('duration_sec') > 86400).count()

    print(f"\n  Duration Categories:")
    print(f"    Negative (ERROR):       {negative:>10,} ({negative/total*100:>6.2f}%)")
    print(f"    < 1 minute:             {under_1min:>10,} ({under_1min/total*100:>6.2f}%)")
    print(f"    Normal (1min-24h):      {normal:>10,} ({normal/total*100:>6.2f}%)")
    print(f"    > 24 hours:             {over_24h:>10,} ({over_24h/total*100:>6.2f}%)")

    # Show examples if outliers exist
    if negative > 0:
        print("\n  ⚠️ Sample NEGATIVE durations (data error!):")
        df_temp.filter(col('duration_sec') < 0) \
               .select('ride_id', 'started_at', 'ended_at', 'duration_sec') \
               .show(3, truncate=False)

    if over_24h > 0:
        print("\n  ⚠️ Sample trips OVER 24 hours:")
        df_temp.filter(col('duration_sec') > 86400) \
               .select('ride_id', 'started_at', 'ended_at', 'duration_sec') \
               .withColumn('duration_hours', col('duration_sec')/3600) \
               .show(5, truncate=False)

    return {
        'negative': negative,
        'under_1min': under_1min,
        'normal': normal,
        'over_24h': over_24h
    }

stats_2023 = analyze_duration(df_2023_new, "2023 (Apr-Dec)")
stats_2024 = analyze_duration(df_2024_all, "2024")


🔍 TRIP DURATION ANALYSIS

📅 2023 (Apr-Dec)
--------------------------------------------------------------------------------

  Duration Statistics:


    count         3,200,875 sec (53,347.9 min)
    mean              2,239 sec (    37.3 min)
    stddev           39,444 sec (   657.4 min)
    min              -3,278 sec (   -54.6 min)
    max           5,902,908 sec (98,381.8 min)



  Duration Categories:
    Negative (ERROR):               91 (  0.00%)
    < 1 minute:                 23,119 (  0.72%)
    Normal (1min-24h):       3,171,859 ( 99.09%)
    > 24 hours:                  5,806 (  0.18%)

  ⚠️ Sample NEGATIVE durations (data error!):
+----------------+-------------------+-------------------+------------+
|ride_id         |started_at         |ended_at           |duration_sec|
+----------------+-------------------+-------------------+------------+
|986CDC95BB7CFB47|2023-11-05 01:54:45|2023-11-05 01:01:08|-3217       |
|4F87FCAECF3ED450|2023-11-05 01:24:23|2023-11-05 01:17:05|-438        |
|C9439097E63D806B|2023-11-05 01:46:17|2023-11-05 01:00:49|-2728       |
+----------------+-------------------+-------------------+------------+
only showing top 3 rows

  ⚠️ Sample trips OVER 24 hours:
+----------------+-------------------+-------------------+------------+------------------+
|ride_id         |started_at         |ended_at           |duration_sec|duration_

    count         4,751,790 sec (79,196.5 min)
    mean              1,035 sec (    17.2 min)
    stddev            3,663 sec (    61.0 min)
    min              -3,063 sec (   -51.0 min)
    max           2,182,940 sec (36,382.3 min)



  Duration Categories:
    Negative (ERROR):               92 (  0.00%)
    < 1 minute:                 17,406 (  0.37%)
    Normal (1min-24h):       4,730,578 ( 99.55%)
    > 24 hours:                  3,714 (  0.08%)

  ⚠️ Sample NEGATIVE durations (data error!):


+----------------+-------------------+-------------------+------------+
|ride_id         |started_at         |ended_at           |duration_sec|
+----------------+-------------------+-------------------+------------+
|42FD9B0288B753FA|2024-05-08 17:02:33|2024-05-08 17:02:32|-1          |
|7DF59206EF093D8F|2024-05-13 01:04:47|2024-05-13 01:04:46|-1          |
|E6C60BCF47082D1D|2024-05-05 13:18:04|2024-05-05 13:18:03|-1          |
+----------------+-------------------+-------------------+------------+
only showing top 3 rows

  ⚠️ Sample trips OVER 24 hours:
+----------------+-----------------------+-----------------------+------------+------------------+
|ride_id         |started_at             |ended_at               |duration_sec|duration_hours    |
+----------------+-----------------------+-----------------------+------------+------------------+
|94380F1217820413|2024-07-08 18:03:29.693|2024-07-09 19:03:24.764|89995       |24.99861111111111 |
|F24761E728F6C551|2024-07-13 16:32:37.836|

In [45]:
# Analyze trip durations
print("\n" + "="*80)
print("🔍 TRIP DURATION ANALYSIS")
print("="*80)

def analyze_duration(df, name):
    print(f"\n📅 {name}")
    print("-"*80)

    # Calculate duration in seconds
    df_temp = df.withColumn(
        'duration_sec',
        unix_timestamp('ended_at') - unix_timestamp('started_at')
    )

    # Statistics
    print("\n  Duration Statistics:")
    duration_stats = df_temp.select('duration_sec').describe().collect()

    for row in duration_stats:
        stat = row[0]
        value_sec = float(row[1]) if row[1] else 0
        value_min = value_sec / 60
        print(f"    {stat:<10} {value_sec:>12,.0f} sec ({value_min:>8,.1f} min)")

    # Categorize
    total = df_temp.count()

    negative = df_temp.filter(col('duration_sec') < 0).count()
    under_1min = df_temp.filter((col('duration_sec') >= 0) & (col('duration_sec') < 60)).count()
    normal = df_temp.filter((col('duration_sec') >= 60) & (col('duration_sec') <= 86400)).count()
    over_24h = df_temp.filter(col('duration_sec') > 86400).count()

    print(f"\n  Duration Categories:")
    print(f"    Negative (ERROR):       {negative:>10,} ({negative/total*100:>6.2f}%)")
    print(f"    < 1 minute:             {under_1min:>10,} ({under_1min/total*100:>6.2f}%)")
    print(f"    Normal (1min-24h):      {normal:>10,} ({normal/total*100:>6.2f}%)")
    print(f"    > 24 hours:             {over_24h:>10,} ({over_24h/total*100:>6.2f}%)")

    # Show examples if outliers exist
    if negative > 0:
        print("\n  ⚠️ Sample NEGATIVE durations (data error!):")
        df_temp.filter(col('duration_sec') < 0) \
               .select('ride_id', 'started_at', 'ended_at', 'duration_sec') \
               .show(3, truncate=False)

    if over_24h > 0:
        print("\n  ⚠️ Sample trips OVER 24 hours:")
        df_temp.filter(col('duration_sec') > 86400) \
               .select('ride_id', 'started_at', 'ended_at', 'duration_sec') \
               .withColumn('duration_hours', col('duration_sec')/3600) \
               .show(5, truncate=False)

    return {
        'negative': negative,
        'under_1min': under_1min,
        'normal': normal,
        'over_24h': over_24h
    }

stats_2023 = analyze_duration(df_2023_new, "2023 (Apr-Dec)")
stats_2024 = analyze_duration(df_2024_all, "2024")


🔍 TRIP DURATION ANALYSIS

📅 2023 (Apr-Dec)
--------------------------------------------------------------------------------

  Duration Statistics:


    count         3,200,875 sec (53,347.9 min)
    mean              2,239 sec (    37.3 min)
    stddev           39,444 sec (   657.4 min)
    min              -3,278 sec (   -54.6 min)
    max           5,902,908 sec (98,381.8 min)



  Duration Categories:
    Negative (ERROR):               91 (  0.00%)
    < 1 minute:                 23,119 (  0.72%)
    Normal (1min-24h):       3,171,859 ( 99.09%)
    > 24 hours:                  5,806 (  0.18%)

  ⚠️ Sample NEGATIVE durations (data error!):
+----------------+-------------------+-------------------+------------+
|ride_id         |started_at         |ended_at           |duration_sec|
+----------------+-------------------+-------------------+------------+
|986CDC95BB7CFB47|2023-11-05 01:54:45|2023-11-05 01:01:08|-3217       |
|4F87FCAECF3ED450|2023-11-05 01:24:23|2023-11-05 01:17:05|-438        |
|C9439097E63D806B|2023-11-05 01:46:17|2023-11-05 01:00:49|-2728       |
+----------------+-------------------+-------------------+------------+
only showing top 3 rows

  ⚠️ Sample trips OVER 24 hours:
+----------------+-------------------+-------------------+------------+------------------+
|ride_id         |started_at         |ended_at           |duration_sec|duration_

    count         4,751,790 sec (79,196.5 min)
    mean              1,035 sec (    17.2 min)
    stddev            3,663 sec (    61.0 min)
    min              -3,063 sec (   -51.0 min)
    max           2,182,940 sec (36,382.3 min)



  Duration Categories:
    Negative (ERROR):               92 (  0.00%)
    < 1 minute:                 17,406 (  0.37%)
    Normal (1min-24h):       4,730,578 ( 99.55%)
    > 24 hours:                  3,714 (  0.08%)

  ⚠️ Sample NEGATIVE durations (data error!):


+----------------+-------------------+-------------------+------------+
|ride_id         |started_at         |ended_at           |duration_sec|
+----------------+-------------------+-------------------+------------+
|42FD9B0288B753FA|2024-05-08 17:02:33|2024-05-08 17:02:32|-1          |
|7DF59206EF093D8F|2024-05-13 01:04:47|2024-05-13 01:04:46|-1          |
|E6C60BCF47082D1D|2024-05-05 13:18:04|2024-05-05 13:18:03|-1          |
+----------------+-------------------+-------------------+------------+
only showing top 3 rows

  ⚠️ Sample trips OVER 24 hours:
+----------------+-----------------------+-----------------------+------------+------------------+
|ride_id         |started_at             |ended_at               |duration_sec|duration_hours    |
+----------------+-----------------------+-----------------------+------------+------------------+
|94380F1217820413|2024-07-08 18:03:29.693|2024-07-09 19:03:24.764|89995       |24.99861111111111 |
|F24761E728F6C551|2024-07-13 16:32:37.836|

In [ ]:
# Calculate what % of data we'll keep after all cleaning
print("="*80)
print("📊 DATA RETENTION ESTIMATE")
print("="*80)

def estimate_cleaning_impact(df, name):
    print(f"\n📅 {name}")
    print("-"*80)

    total = df.count()

    # Calculate duration
    df_temp = df.withColumn(
        'duration_sec',
        unix_timestamp('ended_at') - unix_timestamp('started_at')
    )

    # Apply all filters (simulating cleaning)
    df_clean = df_temp.filter(
        # No nulls in critical fields
        col('ride_id').isNotNull() &
        col('started_at').isNotNull() &
        col('ended_at').isNotNull() &
        col('start_station_id').isNotNull() &
        col('end_station_id').isNotNull() &
        # Valid duration
        (col('duration_sec') >= 60) &
        (col('duration_sec') <= 86400)
    ).dropDuplicates(['ride_id'])

    final = df_clean.count()
    removed = total - final
    retention_rate = (final / total) * 100

    print(f"\n  Original records:     {total:,}")
    print(f"  After cleaning:       {final:,}")
    print(f"  Removed:              {removed:,}")
    print(f"  Retention rate:       {retention_rate:.2f}%")

    return final

clean_2023 = estimate_cleaning_impact(df_2023_new, "2023 (Apr-Dec)")
clean_2024 = estimate_cleaning_impact(df_2024_all, "2024")

print("\n" + "="*80)
print(f"📊 TOTAL CLEANED DATA: {clean_2023 + clean_2024:,} records")
print(f"📊 RETENTION RATE: {((clean_2023 + clean_2024) / (df_2023_new.count() + df_2024_all.count())) * 100:.2f}%")
print("="*80)

📊 DATA RETENTION ESTIMATE

📅 2023 (Apr-Dec)
--------------------------------------------------------------------------------



  Original records:     3,200,875
  After cleaning:       3,157,456
  Removed:              43,419
  Retention rate:       98.64%

📅 2024
--------------------------------------------------------------------------------



  Original records:     4,751,790
  After cleaning:       4,724,527
  Removed:              27,263
  Retention rate:       99.43%

📊 TOTAL CLEANED DATA: 7,881,983 records
📊 RETENTION RATE: 99.11%


In [ ]:
# Final cleaning function based on our analysis
def clean_bluebikes_data(df, year_label="unknown"):
    """
    Clean Bluebikes trip data based on exploration findings

    Cleaning steps:
    1. Remove duplicates by ride_id
    2. Remove rows with missing critical fields
    3. Calculate trip duration
    4. Filter outliers (< 1min or > 24h)
    5. Standardize text fields

    Args:
        df: Raw DataFrame
        year_label: Year identifier for logging
    Returns:
        Cleaned DataFrame
    """

    print(f"\n{'='*60}")
    print(f"🧹 Cleaning {year_label} Data")
    print(f"{'='*60}")

    initial = df.count()
    print(f"📊 Initial records: {initial:,}")

    # Step 1: Remove duplicates
    print("\n1️⃣ Removing duplicates...")
    before = df.count()
    df = df.dropDuplicates(['ride_id'])
    after = df.count()
    print(f"   Removed: {before - after:,} duplicates")

    # Step 2: Calculate trip duration
    print("\n2️⃣ Calculating trip duration...")
    df = df.withColumn(
        'trip_duration_seconds',
        unix_timestamp('ended_at') - unix_timestamp('started_at')
    )
    print("   ✓ Duration calculated")

    # Step 3: Remove rows with missing critical fields
    print("\n3️⃣ Removing missing critical fields...")
    before = df.count()
    df = df.filter(
        col('ride_id').isNotNull() &
        col('started_at').isNotNull() &
        col('ended_at').isNotNull() &
        col('start_station_id').isNotNull() &
        col('end_station_id').isNotNull()
    )
    after = df.count()
    print(f"   Removed: {before - after:,} rows with missing data")

    # Step 4: Filter duration outliers
    print("\n4️⃣ Filtering duration outliers...")
    before = df.count()
    df = df.filter(
        (col('trip_duration_seconds') >= 60) &
        (col('trip_duration_seconds') <= 86400)
    )
    after = df.count()
    print(f"   Removed: {before - after:,} outliers")

    # Step 5: Standardize text fields

In [ ]:
# Final cleaning function based on our analysis
def clean_bluebikes_data(df, year_label="unknown"):
    """
    Clean Bluebikes trip data based on exploration findings

    Cleaning steps:
    1. Remove duplicates by ride_id
    2. Remove rows with missing critical fields
    3. Calculate trip duration
    4. Filter outliers (< 1min or > 24h)
    5. Standardize text fields

    Args:
        df: Raw DataFrame
        year_label: Year identifier for logging
    Returns:
        Cleaned DataFrame
    """

    print(f"\n{'='*60}")
    print(f"🧹 Cleaning {year_label} Data")
    print(f"{'='*60}")

    initial = df.count()
    print(f"📊 Initial records: {initial:,}")

    # Step 1: Remove duplicates
    print("\n1️⃣ Removing duplicates...")
    before = df.count()
    df = df.dropDuplicates(['ride_id'])
    after = df.count()
    print(f"   Removed: {before - after:,} duplicates")

    # Step 2: Calculate trip duration
    print("\n2️⃣ Calculating trip duration...")
    df = df.withColumn(
        'trip_duration_seconds',
        unix_timestamp('ended_at') - unix_timestamp('started_at')
    )
    print("   ✓ Duration calculated")

    # Step 3: Remove rows with missing critical fields
    print("\n3️⃣ Removing missing critical fields...")
    before = df.count()
    df = df.filter(
        col('ride_id').isNotNull() &
        col('started_at').isNotNull() &
        col('ended_at').isNotNull() &
        col('start_station_id').isNotNull() &
        col('end_station_id').isNotNull()
    )
    after = df.count()
    print(f"   Removed: {before - after:,} rows with missing data")

    # Step 4: Filter duration outliers
    print("\n4️⃣ Filtering duration outliers...")
    before = df.count()
    df = df.filter(
        (col('trip_duration_seconds') >= 60) &
        (col('trip_duration_seconds') <= 86400)
    )
    after = df.count()
    print(f"   Removed: {before - after:,} outliers")

    # Step 5: Standardize text fields
    print("\n5️⃣ Standardizing text fields...")
    df = df.withColumn('rideable_type', lower(trim(col('rideable_type'))))
    df = df.withColumn('member_casual', lower(trim(col('member_casual'))))
    print("   ✓ Text standardized")

    # Step 6: Add useful columns
    print("\n6️⃣ Adding derived columns...")
    df = df.withColumn('trip_duration_minutes', col('trip_duration_seconds') / 60)
    df = df.withColumn('start_hour', hour(col('started_at')))
    df = df.withColumn('start_day_of_week', dayofweek(col('started_at')))
    df = df.withColumn('start_month', month(col('started_at')))
    df = df.withColumn('start_year', year(col('started_at')))
    print("   ✓ Time features added")

    final = df.count()
    removed = initial - final
    retention = (final / initial) * 100

    print(f"\n{'='*60}")
    print(f"✅ Cleaning Complete!")
    print(f"{'='*60}")
    print(f"Initial:    {initial:,}")
    print(f"Final:      {final:,}")
    print(f"Removed:    {removed:,} ({100-retention:.2f}%)")
    print(f"Retained:   {retention:.2f}%")
    print(f"{'='*60}\n")

    return df

In [49]:
# Test cleaning on both years
print("="*80)
print("🧪 TESTING CLEANING PIPELINE")
print("="*80)

df_2023_clean = clean_bluebikes_data(df_2023_new, "2023 (Apr-Dec)")
df_2024_clean = clean_bluebikes_data(df_2024_all, "2024")

print(f"\n📊 Total cleaned records: {df_2023_clean.count() + df_2024_clean.count():,}")

🧪 TESTING CLEANING PIPELINE

🧹 Cleaning 2023 (Apr-Dec) Data
📊 Initial records: 3,200,875

1️⃣ Removing duplicates...


   Removed: 0 duplicates

2️⃣ Calculating trip duration...
   ✓ Duration calculated

3️⃣ Removing missing critical fields...


   Removed: 17,248 rows with missing data

4️⃣ Filtering duration outliers...


   Removed: 26,171 outliers

5️⃣ Standardizing text fields...
   ✓ Text standardized

6️⃣ Adding derived columns...
   ✓ Time features added



✅ Cleaning Complete!
Initial:    3,200,875
Final:      3,157,456
Removed:    43,419 (1.36%)
Retained:   98.64%


🧹 Cleaning 2024 Data
📊 Initial records: 4,751,790

1️⃣ Removing duplicates...


   Removed: 141 duplicates

2️⃣ Calculating trip duration...
   ✓ Duration calculated

3️⃣ Removing missing critical fields...


   Removed: 9,523 rows with missing data

4️⃣ Filtering duration outliers...


   Removed: 17,599 outliers

5️⃣ Standardizing text fields...
   ✓ Text standardized

6️⃣ Adding derived columns...
   ✓ Time features added



✅ Cleaning Complete!
Initial:    4,751,790
Final:      4,724,527
Removed:    27,263 (0.57%)
Retained:   99.43%




📊 Total cleaned records: 7,881,983


In [50]:
# Save cleaned data to disk
print("\n" + "="*80)
print("💾 SAVING CLEANED DATA")
print("="*80)

# Define output path
OUTPUT_PATH = os.path.join(BASE_DIR, "data/processed/cleaned")
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Save 2023
print("\n📁 Saving 2023 (Apr-Dec)...")
path_2023 = os.path.join(OUTPUT_PATH, "year=2023")
df_2023_clean.write.mode("overwrite").parquet(path_2023)
print(f"   ✓ Saved to: {path_2023}")

# Save 2024
print("\n📁 Saving 2024...")
path_2024 = os.path.join(OUTPUT_PATH, "year=2024")
df_2024_clean.write.mode("overwrite").parquet(path_2024)
print(f"   ✓ Saved to: {path_2024}")

print("\n" + "="*80)
print("✅ ALL DATA SAVED SUCCESSFULLY!")
print("="*80)
print(f"\n📁 Cleaned data location: {OUTPUT_PATH}")
print("   • year=2023/ (3.2M records)")
print("   • year=2024/ (4.7M records)")
print("\n💡 Use this for feature engineering and model training!")


💾 SAVING CLEANED DATA

📁 Saving 2023 (Apr-Dec)...


   ✓ Saved to: /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/processed/cleaned/year=2023

📁 Saving 2024...


   ✓ Saved to: /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/processed/cleaned/year=2024

✅ ALL DATA SAVED SUCCESSFULLY!

📁 Cleaned data location: /Users/anshumansingh/Machine Learning /bluebikes-demand-predictor/data/processed/cleaned
   • year=2023/ (3.2M records)
   • year=2024/ (4.7M records)

💡 Use this for feature engineering and model training!


In [52]:
# Upload to GCS (single-threaded for Mac compatibility)
print("\n" + "="*80)
print("☁️ UPLOADING TO GOOGLE CLOUD STORAGE")
print("="*80)

import subprocess

GCS_OUTPUT = "gs://bluebikes-demand-predictor-data/processed/cleaned/"

print(f"\n📤 Target: {GCS_OUTPUT}")

# Upload 2023 (no -m flag)
print("\n📁 Uploading 2023...")
subprocess.run(["gsutil", "cp", "-r", path_2023, f"{GCS_OUTPUT}year=2023/"])
print("   ✅ 2023 uploaded")

# Upload 2024
print("\n📁 Uploading 2024...")
subprocess.run(["gsutil", "cp", "-r", path_2024, f"{GCS_OUTPUT}year=2024/"])
print("   ✅ 2024 uploaded")

print("\n✅ DATA SYNCED TO GCS!")


☁️ UPLOADING TO GOOGLE CLOUD STORAGE

📤 Target: gs://bluebikes-demand-predictor-data/processed/cleaned/

📁 Uploading 2023...


FileNotFoundError: [Errno 2] No such file or directory: 'gsutil'

26/01/28 13:57:02 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 869285 ms exceeds timeout 120000 ms
26/01/28 13:57:02 WARN SparkContext: Killing executors is not supported by current scheduler.
26/01/28 13:57:07 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$